In [2]:
import json
import requests
from rdflib import Graph, Namespace, RDF, RDFS
from rdflib.namespace import SKOS
from typing import Dict, List, Optional, Set

class ILCRdfParser:
    """
    Парсер ILC2 на основе rdflib.
    Источник: https://www.iskoi.org/ilc/2/skos/ILC2.ttl
    """
    SKOS_URL = "https://www.iskoi.org/ilc/2/skos/ILC2.ttl"
    ILC_NS = Namespace("http://www.iskoi.org/ilc/2/class/")

    def __init__(self, min_description_length: int = 30, max_depth: int = 3):
        self.min_description_length = min_description_length
        self.max_depth = max_depth
        self.graph = Graph()
        
        # Данные концептов: id -> {notation, label, description}
        self.concepts: Dict[str, Dict] = {}
        self.parents: Dict[str, str] = {}        # child_id -> parent_id
        self.children: Dict[str, List[str]] = {} # parent_id -> [child_id]

    def fetch_and_parse(self):
        """Загружает RDF и извлекает все необходимые данные."""
        print(f"Загрузка RDF из {self.SKOS_URL}")
        resp = requests.get(self.SKOS_URL, timeout=60)
        resp.raise_for_status()
        
        self.graph.parse(data=resp.text, format="turtle")
        print(f"Загружено {len(self.graph)} триплетов")
        
        self._extract_concepts()
        self._extract_labels_and_descriptions()
        self._extract_hierarchy()
        
        print(f"Найдено классов с нотацией: {len(self.concepts)}")
        print(f"Найдено иерархических связей: {len(self.parents)}")

    def _extract_concepts(self):
        """Собирает все ресурсы, имеющие нотацию (skos:notation)."""
        for s, p, o in self.graph.triples((None, SKOS.notation, None)):
            uri = str(s)
            cid = self._extract_id(uri)
            notation = str(o)
            if cid not in self.concepts:
                self.concepts[cid] = {'notation': notation, 'label': '', 'description': ''}
            else:
                self.concepts[cid]['notation'] = notation

    def _extract_labels_and_descriptions(self):
        """Добавляет предпочтительные метки и описания."""
        # prefLabel (SKOS)
        for s, p, o in self.graph.triples((None, SKOS.prefLabel, None)):
            cid = self._extract_id(str(s))
            if cid in self.concepts:
                self.concepts[cid]['label'] = str(o)
        # rdfs:label как запасной вариант
        for s, p, o in self.graph.triples((None, RDFS.label, None)):
            cid = self._extract_id(str(s))
            if cid in self.concepts and not self.concepts[cid]['label']:
                self.concepts[cid]['label'] = str(o)
        
        # scopeNote / definition / note
        for pred in [SKOS.scopeNote, SKOS.definition, SKOS.note]:
            for s, p, o in self.graph.triples((None, pred, None)):
                cid = self._extract_id(str(s))
                if cid in self.concepts and not self.concepts[cid]['description']:
                    self.concepts[cid]['description'] = str(o)

    def _extract_hierarchy(self):
        """Извлекает связи родитель-потомок через skos:broader, rdfs:subClassOf, rdfs:subPropertyOf."""
        # Предикаты, означающие родительскую связь
        hierarchy_predicates = [
            SKOS.broader,
            RDFS.subClassOf,
            RDFS.subPropertyOf
        ]
        for pred in hierarchy_predicates:
            for s, p, o in self.graph.triples((None, pred, None)):
                child_uri = str(s)
                parent_uri = str(o)
                # Принимаем только родителей из пространства имён ILC
                if not parent_uri.startswith(str(self.ILC_NS)):
                    continue
                child_id = self._extract_id(child_uri)
                parent_id = self._extract_id(parent_uri)
                self.parents[child_id] = parent_id
                self.children.setdefault(parent_id, []).append(child_id)
        
        # skos:narrower (обратное направление)
        for s, p, o in self.graph.triples((None, SKOS.narrower, None)):
            parent_uri = str(s)
            child_uri = str(o)
            parent_id = self._extract_id(parent_uri)
            child_id = self._extract_id(child_uri)
            self.parents[child_id] = parent_id
            self.children.setdefault(parent_id, []).append(child_id)
        
        # Удаляем дубликаты детей
        for pid in self.children:
            self.children[pid] = list(set(self.children[pid]))

    @staticmethod
    def _extract_id(uri: str) -> str:
        """Извлекает локальный идентификатор из URI."""
        if '/' in uri:
            return uri.split('/')[-1]
        if '#' in uri:
            return uri.split('#')[-1]
        return uri

    def calculate_depth(self) -> Dict[str, int]:
        """Вычисляет глубину каждого концепта в иерархии."""
        depths = {}
        def get_depth(cid: str, visited: Set[str] = None) -> int:
            if visited is None:
                visited = set()
            if cid in visited or cid in depths:
                return depths.get(cid, 0)
            visited.add(cid)
            parent = self.parents.get(cid)
            if not parent:
                depths[cid] = 0
                return 0
            d = get_depth(parent, visited.copy()) + 1
            depths[cid] = d
            return d
        
        for cid in self.concepts:
            if cid not in depths:
                get_depth(cid)
        return depths

    def has_adequate_description(self, cid: str) -> bool:
        """Проверяет, достаточно ли длинное описание."""
        desc = self.concepts.get(cid, {}).get('description', '')
        return len(desc.strip()) >= self.min_description_length

    def build_tree(self) -> List[Dict]:
        """Строит дерево классификации до указанной глубины."""
        depths = self.calculate_depth()
        all_ids = set(self.concepts.keys())
        root_ids = [cid for cid in all_ids if cid not in self.parents]
        
        def build_node(cid: str, current_depth: int) -> Optional[Dict]:
            if current_depth > self.max_depth:
                return None
            concept = self.concepts.get(cid, {})
            notation = concept.get('notation', '')
            label = concept.get('label', cid)
            description = concept.get('description', '')
            
            # Пропускаем классы без адекватного описания (кроме корневых)
            if current_depth > 0 and not self.has_adequate_description(cid):
                return None
            
            node = {
                'id': cid,
                'notation': notation,
                'label': label,
                'description': description,
                'depth': current_depth,
                'children': []
            }
            
            if current_depth < self.max_depth:
                for child_id in self.children.get(cid, []):
                    child_node = build_node(child_id, current_depth + 1)
                    if child_node:
                        node['children'].append(child_node)
            return node
        
        tree = []
        for rid in root_ids:
            root_node = build_node(rid, 0)
            if root_node and (root_node['children'] or self.has_adequate_description(rid)):
                tree.append(root_node)
        return tree


def save_to_json(tree: List[Dict], filename: str = "ilc_level3.json") -> None:
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(tree, f, ensure_ascii=False, indent=2)
    print(f"Сохранено в {filename}")


def main():
    parser = ILCRdfParser(min_description_length=30, max_depth=3)
    parser.fetch_and_parse()
    tree = parser.build_tree()
    save_to_json(tree)
    print(f"Разделов первого уровня: {len(tree)}")


if __name__ == "__main__":
    main()

Загрузка RDF из https://www.iskoi.org/ilc/2/skos/ILC2.ttl
Загружено 88712 триплетов
Найдено классов с нотацией: 10117
Найдено иерархических связей: 10011
Сохранено в ilc_level3.json
Разделов первого уровня: 21


In [12]:
ROOT_DATA_SOURCE = "../data/source_data"

In [13]:
OUTPUT_PATH = ROOT_DATA_SOURCE + "/ilc_subjects_level3.json"

In [18]:
import os
import json
import requests
from rdflib import Graph, Namespace
from rdflib.namespace import SKOS, RDF


ILC_TTL_URL = "https://www.iskoi.org/ilc/2/skos/ILC2.ttl"
ILC_TTL_FILENAME = "ILC2.ttl"

os.makedirs(ROOT_DATA_SOURCE, exist_ok=True)

# 1. Скачивание файла (если отсутствует)
ttl_filepath = os.path.join(ROOT_DATA_SOURCE, ILC_TTL_FILENAME)
if not os.path.exists(ttl_filepath):
    print(f"Скачивание {ILC_TTL_URL} ...")
    resp = requests.get(ILC_TTL_URL)
    resp.raise_for_status()
    with open(ttl_filepath, 'wb') as f:
        f.write(resp.content)
    print(f"Файл сохранён: {ttl_filepath}")
else:
    print(f"Файл {ttl_filepath} уже существует.")

# 2. Парсинг RDF
g = Graph()
g.parse(ttl_filepath, format="turtle")

# 3. Сбор всех концептов с нотацией
nodes = {}  # uri -> {...}
for concept in g.subjects(RDF.type, SKOS.Concept):
    uri = str(concept)
    notation = None
    for o in g.objects(concept, SKOS.notation):
        notation = str(o)
        break
    if not notation:
        continue

    label = None
    for o in g.objects(concept, SKOS.prefLabel):
        label = str(o)
        break

    definition = None
    for o in g.objects(concept, SKOS.definition):
        definition = str(o)
        break

    nodes[uri] = {
        "uri": uri,
        "notation": notation,
        "label": label or notation,
        "description": definition or "",
        "children": set(),
        "parents": set()
    }

# 4. Связи broader/narrower
for concept, broader in g.subject_objects(SKOS.broader):
    child_uri = str(concept)
    parent_uri = str(broader)
    if child_uri in nodes and parent_uri in nodes:
        nodes[child_uri]["parents"].add(parent_uri)
        nodes[parent_uri]["children"].add(child_uri)

# 5. Определяем 25 основных классов (a-y)
MAIN_NOTATIONS = set(chr(c) for c in range(ord('a'), ord('y')+1))  # a..y
main_root_uris = []
for uri, data in nodes.items():
    if data["notation"] in MAIN_NOTATIONS:
        main_root_uris.append(uri)

print(f"Найдено {len(main_root_uris)} основных классов (a-y): {[nodes[u]['notation'] for u in main_root_uris]}")

# 6. Рекурсивное построение дерева (только от выбранных корней, глубина ≤3)
def build_tree(uri, depth=1, max_depth=3):
    data = nodes[uri]
    node = {
        "notation": data["notation"],
        "label": data["label"],
        "description": data["description"],
        "children": []
    }
    if depth < max_depth:
        for child_uri in sorted(data["children"]):  # сортировка для стабильности
            # Исключаем переход в другие основные классы (если они вдруг оказались потомками)
            if nodes[child_uri]["notation"] in MAIN_NOTATIONS and depth > 1:
                continue  # не включаем основные классы как потомков друг друга на глубине >1
            child_node = build_tree(child_uri, depth + 1, max_depth)
            node["children"].append(child_node)
    return node

# 7. Формируем финальный список
tree = [build_tree(uri) for uri in sorted(main_root_uris, key=lambda u: nodes[u]["notation"])]

# 8. Сохраняем JSON
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(tree, f, ensure_ascii=False, indent=2)

print(f"Дерево из {len(tree)} основных классов (глубина ≤3) сохранено в {OUTPUT_PATH}")

Файл ../data/source_data\ILC2.ttl уже существует.
Найдено 25 основных классов (a-y): ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y']
Дерево из 25 основных классов (глубина ≤3) сохранено в ../data/source_data/ilc_subjects_level3.json


In [16]:
root_uris

['http://www.iskoi.org/ilc/2/class/%7E',
 'http://www.iskoi.org/ilc/2/class/*',
 'http://www.iskoi.org/ilc/2/class/A',
 'http://www.iskoi.org/ilc/2/class/B',
 'http://www.iskoi.org/ilc/2/class/C',
 'http://www.iskoi.org/ilc/2/class/D',
 'http://www.iskoi.org/ilc/2/class/E',
 'http://www.iskoi.org/ilc/2/class/F',
 'http://www.iskoi.org/ilc/2/class/G',
 'http://www.iskoi.org/ilc/2/class/H',
 'http://www.iskoi.org/ilc/2/class/I',
 'http://www.iskoi.org/ilc/2/class/J',
 'http://www.iskoi.org/ilc/2/class/K',
 'http://www.iskoi.org/ilc/2/class/L',
 'http://www.iskoi.org/ilc/2/class/M',
 'http://www.iskoi.org/ilc/2/class/N',
 'http://www.iskoi.org/ilc/2/class/O',
 'http://www.iskoi.org/ilc/2/class/P',
 'http://www.iskoi.org/ilc/2/class/Q',
 'http://www.iskoi.org/ilc/2/class/R',
 'http://www.iskoi.org/ilc/2/class/S',
 'http://www.iskoi.org/ilc/2/class/T',
 'http://www.iskoi.org/ilc/2/class/U',
 'http://www.iskoi.org/ilc/2/class/V',
 'http://www.iskoi.org/ilc/2/class/X',
 'http://www.iskoi.org/